Notebook Description.

According to the files received for the Solar, GAIA1, and GAIA2 portfolios, the table d-cdh.reference.collections_preonboarding has been created.
Following validation and discussions with the business team and Pavlo kotzabougiouka, all the signs for the amounts (field Amount) in the GAIA1 and GAIA2 files have been reversed.
Finally, this table has also been populated with all the records that existed in p-cdh.reference.collections_preonboarding.

The corresponding ticket is DT-453


In [0]:
env = "p"



In [0]:

spark.sql(f"TRUNCATE TABLE `{env}-cdh`.`reference`.`collections_preonboarding`")

spark.sql(f"""
    INSERT INTO `{env}-cdh`.`reference`.`collections_preonboarding`
    SELECT *
    FROM `d-cdh`.`reference`.`collections_preonboarding`
""")


spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS `{env}-cdh-delivery`.`transactions`
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{env}-cdh-delivery`.`transactions`.`collections_preonboarding` (
    PortfolioNameFromReference STRING,
    PortfolioNameFromDexposure STRING,  
    Prefix STRING,
    CDI STRING, 
    CDIWithoutPrefixFromReference STRING,  
    CDIFromReference STRING,   
    ExposurekeyWithoutPrefix STRING,
    Amount DOUBLE,
    PaymentType STRING,        
    PaymentDate TIMESTAMP, 
    PaymentDateMMYY STRING,
    Principal DOUBLE,
    Interest DOUBLE,
    Penalty DOUBLE 
)
USING DELTA
""")

In [0]:
from pyspark.sql.functions import lit

reference_table_df = spark.sql(f"""
SELECT *
FROM `{env}-cdh`.`reference`.`collections_preonboarding`
""")

# renames and drops
source_table_df = reference_table_df \
    .withColumnRenamed("Portfolio", "PortfolioNameFromReference") \
    .withColumnRenamed("CollectionType", "PaymentType") \
    .withColumnRenamed("CollectionPeriod", "PaymentDate") \
    .withColumnRenamed("TransactionDateMMYY", "PaymentDateMMYY") \
    .withColumnRenamed("BorrowerID", "CDIWithoutPrefixFromReference") \
    .withColumnRenamed("DebtorCode", "CDIFromReference") \
    .withColumn("PortfolioNameFromDexposure", lit(None)) \
    .withColumn("CDI", lit(None)) \
    .withColumn("prefix", lit(None)) \
    .drop("Comments", "PortfolioId", "IngestionTimestamp")




In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import lit
from pyspark.sql import Window

dexposure_df = spark.table(f"`{env}-cdh`.derivative.dexposure") \
    .withColumn(
        "rn_business",
        F.row_number().over(Window.partitionBy("ExposureKey").orderBy(F.col("BusinessDate").desc()))
    ) \
    .filter(F.col("rn_business") == 1) \
    .drop("rn_business") \
    .withColumn("match_key", 
        F.when(F.length(F.col("ExposureKey")) > 3, F.substring(F.col("ExposureKey"), 4, 100))
        .otherwise(F.col("ExposureKey"))
    ) \
    .withColumn("rn_match",
        F.row_number().over(Window.partitionBy("match_key").orderBy(F.col("BusinessDate").desc()))
    ) \
    .filter(F.col("rn_match") == 1) \
    .drop("rn_match") \
    .select("ExposureKey", "match_key", "PortfolioName", "CDI") \
    .withColumnRenamed("PortfolioName", "portfolioname_dexp") \
    .withColumnRenamed("CDI", "CDI_dexp")

#  portfolio
portfolio_df = spark.table(f"`{env}-cdh`.primary.portfolio") \
    .select("PortfolioName", "Prefix") \
    .distinct() \
    .withColumnRenamed("PortfolioName", "portfolioname_port") \
    .withColumnRenamed("Prefix", "prefix_port")

# # Join με source_table_df

# Join με source_table_df
final_table_df = source_table_df \
    .join(
        dexposure_df,
        source_table_df["ExposurekeyWithoutPrefix"] == dexposure_df["match_key"],
        "left"
    ) \
    .join(
        portfolio_df,
        F.col("portfolioname_dexp") == F.col("portfolioname_port"),
        "left"
    ) \
    .withColumn("PortfolioNameFromDexposure", F.col("portfolioname_dexp")) \
    .withColumn("CDI", F.col("CDI_dexp")) \
    .withColumn("prefix", F.col("prefix_port")) \
    .drop("ExposureKey", "match_key", "portfolioname_dexp", "CDI_dexp", "portfolioname_port", "prefix_port") \
    .select(
        "PortfolioNameFromReference",
        "PortfolioNameFromDexposure",
        "Prefix",
        "CDI",
        "CDIWithoutPrefixFromReference",
        "CDIFromReference",
        "ExposurekeyWithoutPrefix",
        "Amount",
        "PaymentType",
        "PaymentDate",
        "PaymentDateMMYY",
        "Principal",
        "Interest",
        "Penalty"
    )

In [0]:
#write the target table.
target_table = f"`{env}-cdh-delivery`.transactions.collections_preonboarding"

final_table_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_table)